# Exporting Modelica Models to FMUs

## Overview

This notebook demonstrates how to export Modelica models to FMUs using the `OMPython` library. The process involves identifying the Modelica models, compiling them into FMUs, and organizing the output files.

In [107]:
import sys
import os
from typing import Literal
from pathlib import Path
from shutil import move
from datetime import datetime

from OMPython import ModelicaSystem

REPO_ROOT = Path().cwd()
if REPO_ROOT.name != "SystemSimulation":
    raise RuntimeError("This script must be run from the root of the repository.")

PLATFORM = sys.platform
if PLATFORM.startswith("linux"):
    FMU_EXPORTER_PATH = REPO_ROOT / "demos/ControlledPendulum/artifacts/fmus/linux/"
elif PLATFORM.startswith("win"):
    FMU_EXPORTER_PATH = REPO_ROOT / "demos/ControlledPendulum/artifacts/fmus/windows/"
elif PLATFORM.startswith("darwin"):
    FMU_EXPORTER_PATH = REPO_ROOT / "demos/ControlledPendulum/artifacts/fmus/macos/"
else:
    raise RuntimeError(f"Unsupported platform: {PLATFORM}")
FMU_EXPORTER_PATH.mkdir(parents=True, exist_ok=True)

In [108]:
src_dir = Path.cwd() / "demos/ControlledPendulum/src/modelica/ControlledPendulum/"
main_pkg_name = src_dir.name
main_pkg_path = src_dir /'package.mo'

sub_pkgs = {d.name: d for d in src_dir.iterdir() if d.is_dir()}

In [109]:
composed_model_names = {}
for sub_pkg_name, sub_pkg_path in sub_pkgs.items():
    composed_model_names[sub_pkg_name] = {}
    model_files = sub_pkg_path.glob("*.mo")
    for model_file in model_files:
        if model_file.stem == "package":
            continue
        model_name = model_file.stem
        composed_model_name = f"{main_pkg_name}.{sub_pkg_name}.{model_name}"
        composed_model_names[sub_pkg_name][model_name] = composed_model_name

In [110]:
SOLVER = Literal["cvode", "euler"]
def create_fmu(package_file_path: Path,
               composed_model_name: str,
               solver: SOLVER,
               export_path: Path):
    """Create a ModelicaSystem instance for a given package and model.
    
    Args:
        package_file_path (Path): Path to the Modelica main package file (package.mo).
        composed_model_name (str): Name of the model to be instantiated (e.g., "MainPackageName.SubPackageName.ModelName").
        solver (LiteralString): The solver to be used for simulation (e.g., "cvode", "euler").
    Returns:
        ModelicaSystem: An instance of ModelicaSystem for the specified model.
    """
    modelica_system = ModelicaSystem(
        fileName=str(package_file_path),
        modelName=composed_model_name,
        commandLineOptions=f"--fmiFlags=s:{solver}",
    )

    modelica_system.buildModel()

    fmu_path = modelica_system.convertMo2Fmu(version="2.0", fmuType="cs")
    move(fmu_path, export_path)
    print(f"FMU created at: {export_path}")

In [111]:
# Define Models that shall also be compiled with Euler solver
euler_models = ["PID_Continuous", "Pendulum"]

In [112]:
for sub_pkg_name, model_dict in composed_model_names.items():
    # 1) Create the export directory for the sub-package
    export_dir = FMU_EXPORTER_PATH / sub_pkg_name
    export_dir.mkdir(parents=True, exist_ok=True)

    # 2) Iterate over the models in the sub-package and export each one as an FMU
    for model_name, composed_model_name in model_dict.items():
        print(100 * '=')
        print(f"Creating FMU for model: {composed_model_name}")
        if model_name in euler_models:
            export_path = export_dir / f"{model_name}_cvode.fmu"
            create_fmu(
                package_file_path=main_pkg_path,
                composed_model_name=composed_model_name,
                solver="cvode",
                export_path=export_path,
            )
            export_path = export_dir / f"{model_name}_euler.fmu"
            create_fmu(
                package_file_path=main_pkg_path,
                composed_model_name=composed_model_name,
                solver="euler",
                export_path=export_path,
            )
        else:
            export_path = export_dir / f"{model_name}.fmu"
            create_fmu(
                package_file_path=main_pkg_path,
                composed_model_name=composed_model_name,
                solver="cvode",
                export_path=export_path,
            )
        

Creating FMU for model: ControlledPendulum.Controllers.PID_Continuous

Notification: Automatically loaded package Complex 4.0.0 due to uses annotation from Modelica.
Notification: Automatically loaded package ModelicaServices 4.0.0 due to uses annotation from Modelica.
Notification: Automatically loaded package Modelica 4.0.0 due to usage.


FMU created at: /home/flo/code/SystemSimulation/demos/ControlledPendulum/artifacts/fmus/linux/Controllers/PID_Continuous_cvode.fmu

Notification: Automatically loaded package Complex 4.0.0 due to uses annotation from Modelica.
Notification: Automatically loaded package ModelicaServices 4.0.0 due to uses annotation from Modelica.
Notification: Automatically loaded package Modelica 4.0.0 due to usage.


FMU created at: /home/flo/code/SystemSimulation/demos/ControlledPendulum/artifacts/fmus/linux/Controllers/PID_Continuous_euler.fmu
Creating FMU for model: ControlledPendulum.Controllers.PID_Sampled

Notification: Automatically loaded package Complex 4